# 像素级回归

## 介绍

许多重要的地球科学量，如冠层高度、生物量、土壤湿度和植被指数，都是连续变量而非类别。像素级回归为每个像素预测一个连续值，使用类似于分割的编码器-解码器架构，但使用不同的输出层、损失函数和评估指标。

本教程使用`geoai`包演示完整的回归工作流，从数据准备到训练、评估和推理。案例研究使用Landsat图像与NDVI数据配对来预测植被指数值。

分类告诉您存在什么，分割告诉您在哪里，而回归告诉您每个位置存在多少可测量的量。这种定量输出可直接用于科学分析和政策决策，而不会因离散化连续变量而丢失信息。

## 学习目标

在本教程结束时，您将能够：

- 解释像素级分类和像素级回归之间的区别
- 描述遥感图像连续值预测的实际应用
- 调整编码器-解码器分割架构用于回归任务
- 使用`geoai.create_regression_tiles`准备配对的图像和目标栅格
- 使用`geoai.train_pixel_regressor`训练像素级回归模型
- 使用RMSE、MAE、R平方和残差分析评估回归结果
- 使用重叠切片和混合对大栅格运行推理
- 将训练好的模型应用于新图像进行时间预测

## 理解像素回归

### 分类与回归

在分类中，模型从预定义的离散标签集中选择并使用交叉熵损失。回归预测连续数值（如NDVI为0.65或冠层高度为12.3米）并使用MSE或其变体，惩罚预测值和实际值之间差异的大小。

这种区别在整个管道中具有实际意义：

- **输出层**：分类对N个类别使用softmax激活。回归使用单个输出通道，不使用激活（或使用ReLU强制非负值）。
- **损失函数**：交叉熵变为MSE、MAE或Huber损失。
- **评估指标**：准确性和IoU变为RMSE、MAE和R平方。
- **标签格式**：整数类别掩码变为具有连续值的浮点栅格。

### 应用

像素级回归在地理空间领域具有广泛的适用性：

**植被指数预测。** NDVI和其他光谱指数可以从多光谱图像预测，实现时间间隙填充和跨传感器协调。

**冠层高度估计。** 在激光雷达派生的高度图上训练的模型可以推广到激光雷达不可用的区域，仅从光学图像提供全面的高度估计。

**地上生物量制图。** 回归模型结合光谱波段、植被指数和纹理特征，预测每个像素的生物量密度，用于碳核算和森林管理。

**建筑高度估计。** 在激光雷达派生高度上训练的回归模型可以从广泛可用的航空或卫星图像预测建筑高度，用于城市规划和灾害风险评估。

**土壤特性预测。** 土壤湿度、有机碳、pH值和营养浓度可以从多光谱图像结合地形和气候变量进行估计。

这些应用共享一个共同的工作流：将图像与来自更直接但地理有限的来源的参考测量配对，训练回归模型，并将其应用于在无法直接测量的地方生成全面预测。

## 回归架构

### 调整分割模型用于回归

像U-Net、UNet++、DeepLabV3+和FPN这样的编码器-解码器网络只需最小修改即可用于回归。关键变化在于输出头：用单个输出通道替换多类softmax，不使用激活（或对非负目标使用ReLU）。`geoai`包通过设置`classes=1`自动处理这一点。

### 回归损失函数

选择正确的损失函数会影响训练稳定性和预测质量：

**均方误差（MSE）**由于平方而严重惩罚大误差，使其对异常值敏感，但对最小化最坏情况误差有效。

**平均绝对误差（MAE，L1损失）**对异常值更稳健，但由于梯度恒定，在最优值附近收敛可能较慢。

**Huber损失（平滑L1）**对小误差表现像MSE，对大误差表现像MAE，提供平衡的折中方案。当目标数据可能包含噪声或极值时，它是一个很好的默认选择。

`geoai`包通过`loss_type`参数支持这三种损失函数：`"mse"`、`"l1"`和`"huber"`。

### 输出激活和缩放

对于许多地理空间任务，目标变量具有已知的物理范围。您可以通过两种方式结合此领域知识：

1. **目标范围过滤**：在切片创建期间，使用`target_min`和`target_max`将值裁剪到有效范围，确保干净的训练数据。
2. **预测后裁剪**：在推理期间使用`predict_raster`中的`clip_range`参数将输出值裁剪到有效范围。

实际上，结合两种方法效果很好。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install "geoai-py[extra]"

## 案例研究：从Landsat图像预测NDVI

本案例研究训练一个模型从Landsat图像在像素级预测NDVI。模型在田纳西州诺克斯维尔的2022年数据上训练，然后应用于2023年图像以展示时间泛化能力。

### 环境设置

In [ ]:
import geoai
from sklearn.model_selection import train_test_split

### 下载数据

下载2022年（训练）和2023年（测试）的Landsat图像和NDVI栅格。

In [ ]:
train_raster = geoai.download_file(
    "https://data.source.coop/opengeos/geoai/tn_landsat_2022.tif"
)
train_target = geoai.download_file(
    "https://data.source.coop/opengeos/geoai/tn_ndvi_2022.tif"
)
test_raster = geoai.download_file(
    "https://data.source.coop/opengeos/geoai/tn_landsat_2023.tif"
)

### 检查数据

检查输入和目标栅格以了解数据维度和值范围。

In [ ]:
import rasterio

with rasterio.open(train_raster) as src:
    in_channels = src.count
    print(f"Input shape: {src.height} x {src.width}, {src.count} bands")
    print(f"Input CRS: {src.crs}")
    print(f"Input resolution: {src.res}")

with rasterio.open(train_target) as src:
    print(f"Target shape: {src.height} x {src.width}, {src.count} band(s)")
    target_data = src.read(1)
    print(f"Target value range: [{target_data.min():.2f}, {target_data.max():.2f}]")

### 创建训练切片

将大栅格切成较小的补丁用于训练。NDVI值裁剪到[-1, 1]，过滤掉nodata像素过多的切片，50%重叠（`tile_size=256`时`stride=128`）增加训练样本。

In [ ]:
image_paths, target_paths = geoai.create_regression_tiles(
    input_raster=train_raster,
    target_raster=train_target,
    output_dir="ndvi_tiles",
    tile_size=256,
    stride=128,
    target_band=1,
    min_valid_ratio=0.9,
    target_min=-1.0,
    target_max=1.0,
)
print(f"Created {len(image_paths)} tiles")

### 分割数据

将切片分割为80%训练集和20%验证集。

In [ ]:
train_imgs, val_imgs, train_tgts, val_tgts = train_test_split(
    image_paths, target_paths, test_size=0.2, random_state=42
)
print(f"Training: {len(train_imgs)}, Validation: {len(val_imgs)}")

### 训练模型

训练一个带有ResNet34编码器的U-Net模型用于像素级NDVI回归。该函数处理模型创建、数据加载、使用AdamW优化器训练、学习率调度和提前停止。

In [ ]:
model = geoai.train_pixel_regressor(
    train_image_paths=train_imgs,
    train_target_paths=train_tgts,
    val_image_paths=val_imgs,
    val_target_paths=val_tgts,
    encoder_name="resnet34",
    architecture="unet",
    in_channels=in_channels,
    output_dir="ndvi_model",
    batch_size=8,
    num_epochs=100,
    learning_rate=1e-3,
    num_workers=0,
    loss_type="mse",
    patience=20,
    devices=1,
    verbose=False,
)

### 监控训练历史

绘制训练和验证损失以及R平方曲线来诊断模型行为。两个损失应该一起下降，R平方应该向1.0增加。

In [ ]:
fig, history_df = geoai.plot_training_history(
    log_dir="ndvi_model",
    metrics=["loss", "r2"],
)

### 在训练区域运行推理

在2022年栅格上运行推理以对照真实值进行评估。该函数使用带重叠的滑动窗口和高斯加权混合实现无缝预测。

In [ ]:
geoai.predict_raster(
    model=model,
    input_raster=train_raster,
    output_raster="ndvi_model/predicted_ndvi_2022.tif",
    tile_size=256,
    overlap=64,
    batch_size=8,
    clip_range=(-1.0, 1.0),
)

### 评估结果

比较预测的NDVI与真实值。该函数并排生成真实值、预测值和残差图。

In [ ]:
fig, metrics = geoai.plot_regression_comparison(
    true_raster=train_target,
    pred_raster="ndvi_model/predicted_ndvi_2022.tif",
    title="NDVI Prediction Results",
    cmap="RdYlGn",
    vmin=-0.2,
    vmax=0.8,
    valid_range=(-1.0, 1.0),
)

创建预测值与实际值的散点图以揭示系统偏差。点应聚集在1:1线上。

In [ ]:
fig, metrics = geoai.plot_scatter(
    true_raster=train_target,
    pred_raster="ndvi_model/predicted_ndvi_2022.tif",
    sample_size=50000,
    valid_range=(-1.0, 1.0),
    fit_line=True,
)

### 在新数据上预测（2023年）

将训练好的模型应用于2023年Landsat图像以展示时间泛化能力。

In [ ]:
geoai.predict_raster(
    model=model,
    input_raster=test_raster,
    output_raster="ndvi_model/predicted_ndvi_2023.tif",
    tile_size=256,
    overlap=64,
    batch_size=8,
    clip_range=(-1.0, 1.0),
)

将2023年输入图像与预测的NDVI图一起可视化。

In [ ]:
geoai.visualize_prediction(
    input_raster=test_raster,
    pred_raster="ndvi_model/predicted_ndvi_2023.tif",
    cmap="RdYlGn",
    vmin=-0.2,
    vmax=0.8,
)

## 评估指标

回归模型使用与分类模型不同的指标：

**均方根误差（RMSE）**以与目标变量相同的单位测量预测误差标准差。

**平均绝对误差（MAE）**是平均绝对预测误差，比RMSE对异常值不敏感。

**R平方**测量模型解释的方差比例，接近1.0的值表示性能良好。

**皮尔逊相关**测量预测和目标之间的线性关系，捕捉模型是否正确排名像素（即使存在系统偏差）。

检查残差图可以揭示模型误差的地理模式，例如水体附近或阴影区域的较高误差。

## 关键要点

1. **像素级回归**为每个像素预测连续值，与分配离散标签的分类和分割不同。

2. **编码器-解码器架构**通过将输出更改为单个通道并切换损失函数，可以直接从分割迁移到回归。

3. **损失函数选择很重要**：MSE惩罚大误差，MAE对异常值稳健，Huber损失提供平衡的折中方案。

4. **数据质量至关重要**：在切片期间使用`target_min`/`target_max`，在推理期间使用`clip_range`来强制有效范围。

5. **`geoai`包**通过`create_regression_tiles`、`train_pixel_regressor`、`predict_raster`和评估函数提供简化的工作流。

6. **评估使用回归特定指标**：RMSE、MAE、R平方和相关性，辅以残差图和散点图。

7. **带混合的重叠切片**生成无缝预测图，没有可见的切片边界伪影。

8. **时间泛化**在模型学习到稳健的特征-目标映射关系后即可实现，由此能够开展时序分析与缺值填补工作。
